# Phase 0: Phenotype & Covariate File Generation for REGENIE

**Last updated:** 2026-02-23  
**Plan reference:** `phase0-phenotype-covariate-generation-f93f49.md`

Generates two REGENIE-formatted files:
- `MS_phenotype.txt` — NS_326.1 (Multiple Sclerosis) case/control status
- `MS_covariates.txt` — Age, Sex, PC1-PC10

Both files are space-delimited with `FID IID` as the first two columns.  
Missing values are encoded as `NA` (regenie standard, **not** `-9`).

## Data Sources
| Source | Path / Table |
|---|---|
| Ancestry PCs + EUR labels | `gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/ancestry/ancestry_preds.tsv` |
| Case/control status | BigQuery: `condition_occurrence` + `person` (via `WORKSPACE_CDR`) |
| Sex + year of birth | BigQuery: `person` table |

## REGENIE Format Requirements
- **Phenotype:** `FID IID <pheno>` — binary coded `1=case, 0=control, NA=missing`
- **Covariates:** `FID IID Age Sex PC1...PC10` — no missing values permitted
- FID = IID = `research_id` (AoU single-ID system)

## Test Mode
Set `TEST_MODE = True` to run on a small random subset (n=500) for fast iteration.

---
## 0. Setup & Configuration

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
from google.cloud import bigquery

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
TEST_MODE = True          # Set False for full production run
TEST_N    = 500           # Number of EUR samples to use in test mode
RANDOM_SEED = 42

# AoU environment variables (set automatically on Researcher Workbench)
WORKSPACE_BUCKET = os.environ["WORKSPACE_BUCKET"]
GOOGLE_PROJECT   = os.environ["GOOGLE_PROJECT"]
WORKSPACE_CDR    = os.environ["WORKSPACE_CDR"]   # BigQuery dataset, e.g. 'aou-res-curation-output-prod.C2022Q4R9'

# Paths
ANCESTRY_PREDS_PATH = "gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/ancestry/ancestry_preds.tsv"
OUTPUT_DIR_GCS      = f"{WORKSPACE_BUCKET}/results/0-phenotype"
OUTPUT_DIR_LOCAL    = "./phase0_outputs"  # local staging before GCS upload

# ICD-10 concept IDs for Multiple Sclerosis (G35)
# OMOP concept_id 374919 = Multiple sclerosis (standard SNOMED)
# We query both ICD-10 source codes and standard OMOP concept IDs for completeness
MS_CONCEPT_IDS  = [374919]          # Standard OMOP concept_id for MS
MS_ICD10_CODES  = ["G35"]           # ICD-10-CM source code

# Age reference: AoU v8 data freeze year
AGE_REFERENCE_YEAR = 2024

os.makedirs(OUTPUT_DIR_LOCAL, exist_ok=True)

print(f"Workspace bucket : {WORKSPACE_BUCKET}")
print(f"CDR dataset      : {WORKSPACE_CDR}")
print(f"Test mode        : {TEST_MODE} (n={TEST_N if TEST_MODE else 'ALL'})")

---
## 1. Load EUR Sample IDs + Ancestry PCs

The `ancestry_preds.tsv` file contains:
- `research_id` — sample identifier (matches PLINK `.fam` IID from Phase 1)
- `ancestry_pred` — predicted ancestry label (filter to `'eur'`)
- `pca_features_0` ... `pca_features_9` — top 10 ancestry PCs

We copy the file locally first (requester-pays bucket).

In [ ]:
print("Copying ancestry_preds.tsv from CDR bucket (requester-pays)...")
!gsutil -u $GOOGLE_PROJECT cp {ANCESTRY_PREDS_PATH} ./ancestry_preds_temp.tsv

ancestry_df = pd.read_csv("./ancestry_preds_temp.tsv", sep="\t", low_memory=False)
!rm ./ancestry_preds_temp.tsv

print(f"Total samples in ancestry file : {len(ancestry_df):,}")
print(f"Ancestry distribution:\n{ancestry_df['ancestry_pred'].value_counts()}")
print(f"\nColumns: {list(ancestry_df.columns)}")

In [ ]:
import ast

# 1. Filter to EUR samples
eur_df = ancestry_df[ancestry_df["ancestry_pred"] == "eur"].copy()
eur_df["research_id"] = eur_df["research_id"].astype(str)
print(f"EUR samples: {len(eur_df):,}")

# 2. Parse the 'pca_features' string/list into individual PC columns
def parse_pcs(val):
    if isinstance(val, str):
        # Convert "[0.1, 0.2, ...]" string to list
        return ast.literal_eval(val)
    return val

print("Parsing pca_features array...")
pc_list = eur_df["pca_features"].apply(parse_pcs).tolist()
pc_matrix = np.array(pc_list)

# 3. Create PC1..PC10 columns
pc_cols = [f"PC{i}" for i in range(1, 11)]
for i in range(10):
    eur_df[f"PC{i+1}"] = pc_matrix[:, i]

# 4. Subset to needed columns
eur_df = eur_df[["research_id"] + pc_cols].copy()

if TEST_MODE:
    eur_df = eur_df.sample(n=min(TEST_N, len(eur_df)), random_state=RANDOM_SEED).copy()
    print(f"[TEST MODE] Subsampled to {len(eur_df):,}")

eur_ids = set(eur_df["research_id"])
print(f"Final EUR sample set size: {len(eur_ids):,}")
eur_df.head(3)

---
## 2. Query Demographics from BigQuery (Age + Sex)

The `person` table contains `person_id`, `year_of_birth`, and `gender_concept_id`.
- `gender_concept_id` 8507 = Male → coded `1`
- `gender_concept_id` 8532 = Female → coded `2`
- Other/unknown → `NA` (sample will be dropped from covariate file)

Note: `person_id` in CDR = `research_id` in ancestry file.

In [ ]:
bq_client = bigquery.Client(project=GOOGLE_PROJECT)

demographics_query = f"""
SELECT
    CAST(p.person_id AS STRING)  AS research_id,
    p.year_of_birth,
    p.gender_concept_id
FROM
    `{WORKSPACE_CDR}.person` AS p
"""

print("Querying person table from BigQuery...")
person_df = bq_client.query(demographics_query).to_dataframe()
person_df["research_id"] = person_df["research_id"].astype(str)

print(f"Person table rows: {len(person_df):,}")
print(f"Gender concept IDs: {person_df['gender_concept_id'].value_counts().to_dict()}")
person_df.head(3)

In [ ]:
# Compute Age and recode Sex
person_df["Age"] = AGE_REFERENCE_YEAR - person_df["year_of_birth"]

# Sex: 8507=Male->1, 8532=Female->2, else NA
sex_map = {8507: 1, 8532: 2}
person_df["Sex"] = person_df["gender_concept_id"].map(sex_map)  # unmapped -> NaN

n_unknown_sex = person_df["Sex"].isna().sum()
print(f"Samples with unknown/other sex (will be excluded): {n_unknown_sex:,}")
print(f"Sex distribution (1=M, 2=F):\n{person_df['Sex'].value_counts(dropna=False)}")

person_df = person_df[["research_id", "Age", "Sex"]].copy()
person_df.head(3)

---
## 3. Query Case/Control Status from BigQuery

NS_326.1 = Multiple Sclerosis.  
We identify cases using:
1. Standard OMOP concept_id `374919` (Multiple sclerosis) in `condition_occurrence`
2. ICD-10-CM source code `G35` as a fallback via `condition_source_value`

Controls = all EUR WGS samples not in the case set.

In [ ]:
# Build concept ID list for SQL
concept_id_list = ", ".join(str(c) for c in MS_CONCEPT_IDS)
icd_code_list   = ", ".join(f"'{c}'" for c in MS_ICD10_CODES)

case_query = f"""
SELECT DISTINCT
    CAST(co.person_id AS STRING) AS research_id
FROM
    `{WORKSPACE_CDR}.condition_occurrence` AS co
WHERE
    co.condition_concept_id IN ({concept_id_list})
    OR co.condition_source_value IN ({icd_code_list})
"""

print("Querying condition_occurrence for MS cases...")
cases_df = bq_client.query(case_query).to_dataframe()
cases_df["research_id"] = cases_df["research_id"].astype(str)

print(f"Total MS cases in CDR (all ancestries): {len(cases_df):,}")

# Restrict to EUR samples
eur_cases = set(cases_df["research_id"]) & eur_ids
eur_controls = eur_ids - eur_cases

print(f"EUR MS cases    : {len(eur_cases):,}")
print(f"EUR controls    : {len(eur_controls):,}")
print(f"EUR total       : {len(eur_ids):,}")

---
## 4. Merge All Data & Build Output DataFrames

In [ ]:
# Build phenotype series: 1=case, 0=control
pheno_records = []
for rid in eur_ids:
    pheno_records.append({"research_id": rid, "MS": 1 if rid in eur_cases else 0})

pheno_df = pd.DataFrame(pheno_records)
pheno_df["research_id"] = pheno_df["research_id"].astype(str)

print(f"Phenotype records: {len(pheno_df):,}")
print(f"Case/control split:\n{pheno_df['MS'].value_counts()}")

In [ ]:
# Merge: EUR PCs + demographics (inner join to keep only samples with all data)
merged_df = eur_df.merge(person_df, on="research_id", how="inner")
merged_df = merged_df.merge(pheno_df, on="research_id", how="inner")

print(f"After merging EUR + demographics + phenotype: {len(merged_df):,} samples")

# Drop samples with missing Sex or Age
n_before = len(merged_df)
merged_df = merged_df.dropna(subset=["Age", "Sex"] + pc_cols)
n_after = len(merged_df)
print(f"Dropped {n_before - n_after:,} samples with missing Age/Sex/PC values")
print(f"Final sample count: {n_after:,}")

# Cast Sex to int
merged_df["Sex"] = merged_df["Sex"].astype(int)
merged_df["Age"] = merged_df["Age"].astype(int)

merged_df.head(3)

---
## 5. Format REGENIE Output Files

Both files:
- Space-delimited
- Header: `FID IID ...`
- `FID = IID = research_id` (AoU single-ID system)
- Missing = `NA`

In [ ]:
# Add FID and IID columns (both = research_id)
merged_df.insert(0, "FID", merged_df["research_id"])
merged_df.insert(1, "IID", merged_df["research_id"])

# --- Phenotype file ---
pheno_cols = ["FID", "IID", "MS"]
pheno_out = merged_df[pheno_cols].copy()

# --- Covariate file ---
covar_cols = ["FID", "IID", "Age", "Sex"] + pc_cols
covar_out = merged_df[covar_cols].copy()

print("Phenotype file preview:")
print(pheno_out.head(5).to_string(index=False))
print(f"\nShape: {pheno_out.shape}")

print("\nCovariate file preview:")
print(covar_out.head(3).to_string(index=False))
print(f"\nShape: {covar_out.shape}")

---
## 6. Validation Checks

In [ ]:
print("=" * 60)
print("VALIDATION CHECKS")
print("=" * 60)

errors = []

# 1. No duplicate IIDs
n_dup_pheno = pheno_out["IID"].duplicated().sum()
n_dup_covar = covar_out["IID"].duplicated().sum()
print(f"[{'PASS' if n_dup_pheno == 0 else 'FAIL'}] Duplicate IIDs in phenotype file: {n_dup_pheno}")
print(f"[{'PASS' if n_dup_covar == 0 else 'FAIL'}] Duplicate IIDs in covariate file : {n_dup_covar}")
if n_dup_pheno > 0: errors.append("Duplicate IIDs in phenotype file")
if n_dup_covar > 0: errors.append("Duplicate IIDs in covariate file")

# 2. Phenotype values are only {0, 1, NA}
valid_pheno_vals = {0, 1}
actual_pheno_vals = set(pheno_out["MS"].dropna().unique())
pheno_ok = actual_pheno_vals.issubset(valid_pheno_vals)
print(f"[{'PASS' if pheno_ok else 'FAIL'}] Phenotype values: {actual_pheno_vals} (expected subset of {{0, 1}})")
if not pheno_ok: errors.append(f"Unexpected phenotype values: {actual_pheno_vals}")

# 3. No missing values in covariate columns
covar_missing = covar_out[covar_cols[2:]].isna().sum().sum()
print(f"[{'PASS' if covar_missing == 0 else 'FAIL'}] Missing values in covariate columns: {covar_missing}")
if covar_missing > 0: errors.append(f"Missing values in covariates: {covar_missing}")

# 4. FID == IID
fid_iid_match = (pheno_out["FID"] == pheno_out["IID"]).all()
print(f"[{'PASS' if fid_iid_match else 'FAIL'}] FID == IID in phenotype file")
if not fid_iid_match: errors.append("FID != IID in phenotype file")

# 5. Sample counts
n_cases    = (pheno_out["MS"] == 1).sum()
n_controls = (pheno_out["MS"] == 0).sum()
n_missing  = pheno_out["MS"].isna().sum()
print(f"\nSample summary:")
print(f"  Cases    : {n_cases:,}")
print(f"  Controls : {n_controls:,}")
print(f"  Missing  : {n_missing:,}")
print(f"  Total    : {len(pheno_out):,}")

# 6. PC range sanity check (PCs should be roughly -1 to 1 or similar scale)
pc_max = covar_out[pc_cols].abs().max().max()
print(f"\nMax absolute PC value: {pc_max:.4f} (sanity check - should be < 100)")

print("\n" + "=" * 60)
if errors:
    print(f"VALIDATION FAILED with {len(errors)} error(s):")
    for e in errors:
        print(f"  - {e}")
else:
    print("ALL VALIDATION CHECKS PASSED")
print("=" * 60)

---
## 7. Write Output Files & Upload to GCS

In [ ]:
# Determine output file suffix for test mode
suffix = "_TEST" if TEST_MODE else ""

pheno_local = os.path.join(OUTPUT_DIR_LOCAL, f"MS_phenotype{suffix}.txt")
covar_local = os.path.join(OUTPUT_DIR_LOCAL, f"MS_covariates{suffix}.txt")

# Write space-delimited files (NA for missing)
pheno_out.to_csv(pheno_local, sep=" ", index=False, na_rep="NA")
covar_out.to_csv(covar_local, sep=" ", index=False, na_rep="NA")

print(f"Written locally:")
print(f"  {pheno_local}")
print(f"  {covar_local}")

# Verify file format by reading back first 5 lines
print("\nPhenotype file (first 5 lines):")
with open(pheno_local) as f:
    for i, line in enumerate(f):
        if i >= 5: break
        print(f"  {line.rstrip()}")

print("\nCovariate file (first 3 lines):")
with open(covar_local) as f:
    for i, line in enumerate(f):
        if i >= 3: break
        print(f"  {line.rstrip()}")

In [ ]:
# Upload to GCS workspace bucket
# Skip upload in test mode unless explicitly confirmed
if not TEST_MODE:
    print(f"Uploading to GCS: {OUTPUT_DIR_GCS}/")
    !gsutil cp {pheno_local} {OUTPUT_DIR_GCS}/MS_phenotype.txt
    !gsutil cp {covar_local} {OUTPUT_DIR_GCS}/MS_covariates.txt
    print("Upload complete.")
    print(f"  {OUTPUT_DIR_GCS}/MS_phenotype.txt")
    print(f"  {OUTPUT_DIR_GCS}/MS_covariates.txt")
else:
    print("[TEST MODE] Skipping GCS upload. Set TEST_MODE=False and re-run to upload production files.")

---
## 8. Inspect Ancestry TSV Column Names (Diagnostic Cell)

Run this cell first if the PC column names are unexpected.  
It prints all columns in `ancestry_preds.tsv` so you can verify the PC column naming convention.

In [ ]:
# Diagnostic: inspect ancestry_preds.tsv columns and first row
# Run this if PC columns are not found or named differently

print("Copying ancestry_preds.tsv for column inspection...")
!gsutil -u $GOOGLE_PROJECT cp {ANCESTRY_PREDS_PATH} ./ancestry_preds_diag.tsv

diag_df = pd.read_csv("./ancestry_preds_diag.tsv", sep="\t", nrows=5)
!rm ./ancestry_preds_diag.tsv

print(f"Columns ({len(diag_df.columns)}):")
for col in diag_df.columns:
    print(f"  {col}: {diag_df[col].dtype} | example: {diag_df[col].iloc[0]}")

---
## 9. Inspect BigQuery Tables (Diagnostic Cell)

Run this cell to verify the CDR table schema and confirm column names before the main query.

In [ ]:
# Diagnostic: preview person table schema
person_preview_query = f"""
SELECT *
FROM `{WORKSPACE_CDR}.person`
LIMIT 3
"""
person_preview = bq_client.query(person_preview_query).to_dataframe()
print("person table columns:")
for col in person_preview.columns:
    print(f"  {col}: {person_preview[col].dtype}")
person_preview.head(3)

In [ ]:
# Diagnostic: preview condition_occurrence table and check MS concept IDs
condition_preview_query = f"""
SELECT
    condition_concept_id,
    condition_source_value,
    COUNT(*) AS n_records
FROM `{WORKSPACE_CDR}.condition_occurrence`
WHERE
    condition_concept_id IN ({concept_id_list})
    OR condition_source_value IN ({icd_code_list})
GROUP BY 1, 2
ORDER BY n_records DESC
LIMIT 20
"""
condition_preview = bq_client.query(condition_preview_query).to_dataframe()
print("MS condition records found:")
print(condition_preview.to_string(index=False))

---
## Summary

| Output File | Description |
|---|---|
| `MS_phenotype.txt` | Space-delimited; `FID IID MS`; 1=case, 0=control, NA=missing |
| `MS_covariates.txt` | Space-delimited; `FID IID Age Sex PC1...PC10`; no missing values |

**REGENIE Step 1 usage:**
```bash
regenie \
  --step 1 \
  --bed results/1-bg_snp/plink_qc/all_background_final_qc \
  --phenoFile results/0-phenotype/MS_phenotype.txt \
  --covarFile results/0-phenotype/MS_covariates.txt \
  --covarCol Age --covarCol Sex --covarCol PC{1:10} \
  --bt \
  --bsize 1000 \
  --lowmem \
  --out results/1-bg_snp/regenie/MS_null_model
```

**Open questions to resolve before production run:**
1. Confirm ICD-10 `G35` / OMOP concept `374919` captures all MS cases in this CDR version.
2. Confirm `AGE_REFERENCE_YEAR = 2024` is appropriate for v8 data freeze.
3. Confirm sample universe: all EUR WGS samples vs. only those in Phase 1 `.fam` file.